In [1]:
import os
import numpy as np
import pandas as pd

patient_id_col = 'PatientID'
patient_id_length = 7
split_col = 'Split'
path_dataset = r'D:\Github\dl_ntcp_multitox\datasets\dataset'
endpoint_cols = ['Xerostomia_M12', 'Dysphagia_6MO', 'Taste_M06']
columns_dict = {
    'xerostomia_stratified_sampling_test_manual_94.csv': [patient_id_col, split_col, 'Xerostomia_M12', 'Xerostomia_W01', 
                                                       'Submandibular_meandose', 'Parotid_meandose_adj',
                                                       'Xerostomia_W01_not_at_all', 'Xerostomia_W01_little', 
                                                       'Xerostomia_W01_moderate_to_severe', 'Gender', 'Age',
                                                         'CT+C_available', 'CT_Artefact', 'Photons', 'Loctum2_v2'],
    'dysphagia_stratified_sampling_test_v3.csv': [patient_id_col, split_col, 'Dysphagia_6MO', 
                                                  'PCM_Sup_meandose', 'PCM_Med_meandose', 'PCM_Inf_meandose', 'Photons'],   
    'taste_features.csv': [patient_id_col, split_col, 'Taste_M06', 'Taste_W01', 'OralCavity_Ext_meandose', 'Photons'] 
}

features_dl = ['Photons']

df = None
for endpoint_file, cols in columns_dict.items():
    df_i = pd.read_csv(os.path.join(path_dataset, endpoint_file), sep=';')
    df_i = df_i[cols]
    
    df_i[patient_id_col] = ['%0.{}d'.format(patient_id_length) % int(x) for x in df_i[patient_id_col]]
    
    for f in features_dl:
        df_i[f] = df_i[f].astype(float) 
    
    # Merge dataframes
    if df is None:
        df = df_i
    else:
        df = df.merge(df_i, on=[patient_id_col, split_col] + features_dl, how='outer')
        
# Fill -1 for missing endpoint values
df[endpoint_cols] = df[endpoint_cols].fillna(value=-1)
        
# Check that there are no duplicated PatientID (this could happen if the duplicated PatientID has different value
# for `split_col` across the different endpoint files
if len(df) != len(set(df[patient_id_col])):
    raise ValueError('There are at least two rows with the same PatientID!')
    
    

In [2]:
df.dtypes

PatientID                             object
Split                                 object
Xerostomia_M12                       float64
Xerostomia_W01                       float64
Submandibular_meandose               float64
Parotid_meandose_adj                 float64
Xerostomia_W01_not_at_all            float64
Xerostomia_W01_little                float64
Xerostomia_W01_moderate_to_severe    float64
Gender                               float64
Age                                  float64
CT+C_available                       float64
CT_Artefact                          float64
Photons                              float64
Loctum2_v2                            object
Dysphagia_6MO                        float64
PCM_Sup_meandose                     float64
PCM_Med_meandose                     float64
PCM_Inf_meandose                     float64
Taste_M06                            float64
Taste_W01                            float64
OralCavity_Ext_meandose              float64
dtype: obj

In [3]:
df.to_csv(os.path.join(path_dataset, 'stratified_sampling_full_all_endpoints.csv'), sep=';', index=False)